# 自适应检索（Adaptive Retrieval）：不同问题用不同检索策略

普通 RAG 往往是固定策略：**一个问题 → 向量检索 top-k → LLM 回答**。但现实里问题类型不同，最合适的检索方式也不同：

- **事实型（Factual）**：需要精确命中；适合“改写查询 + 精排”
- **分析型（Analytical）**：需要覆盖多个方面；适合“拆子问题 + 汇总去重 + 多样性选择”
- **观点型（Opinion）**：需要多视角；适合“生成视角/立场 → 按视角检索 → 选代表性片段”
- **上下文型（Contextual）**：要结合用户背景；适合“把用户上下文注入查询/排序”

这一节我们实现一个可运行的 **Adaptive Retrieval RAG**：

1. 用 LLM 把问题分类到 4 种类型
2. 按类型选择不同的检索策略得到 `docs`
3. 把 `docs` 拼接成 `context`，再让 LLM 生成答案


## 流程图

<div style="text-align: center;">

<img src="../images/adaptive_retrieval.svg" alt="adaptive retrieval" style="width:100%; height:auto;">
</div>

## 1) 导入依赖与环境

本 notebook 不包含安装单元格。请确保环境里已有：`langchain-core / langchain-openai / langchain-text-splitters / langchain-community / pypdf / python-dotenv / requests`。

In [2]:
from __future__ import annotations

import os
import re
import sys
from pathlib import Path
from typing import Iterable, Literal, Optional

import requests
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from pypdf import PdfReader

load_dotenv("../.env")
sys.path.insert(0, str(Path("..").resolve()))

from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS

## 2) 准备语料：下载 PDF（若不存在）→ 抽取文本 → 切分 chunks

In [3]:
os.environ["HTTP_PROXY"] = os.getenv("HTTP_PROXY", "http://127.0.0.1:7890")
os.environ["HTTPS_PROXY"] = os.getenv("HTTPS_PROXY", "http://127.0.0.1:7890")
os.environ["ALL_PROXY"] = os.getenv("ALL_PROXY", "http://127.0.0.1:7890")

In [4]:
DATA_DIR = Path("../data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

PDF_PATH = DATA_DIR / "Understanding_Climate_Change.pdf"
PDF_URL = "https://raw.githubusercontent.com/NirDiamant/RAG_TECHNIQUES/main/data/Understanding_Climate_Change.pdf"


def download_file(url: str, path: Path, timeout_s: int = 60) -> None:
    if path.exists() and path.stat().st_size > 0:
        return
    headers = {"User-Agent": "paper-qa-rag-techniques/1.0"}
    r = requests.get(url, headers=headers, timeout=timeout_s)
    r.raise_for_status()
    path.write_bytes(r.content)


download_file(PDF_URL, PDF_PATH)
PDF_PATH

PosixPath('../data/Understanding_Climate_Change.pdf')

In [5]:
def read_pdf_pages(path: Path) -> list[Document]:
    reader = PdfReader(str(path))
    out: list[Document] = []
    for i, page in enumerate(reader.pages):
        text = page.extract_text() or ""
        text = re.sub(r"\s+", " ", text).strip()
        if not text:
            continue
        out.append(
            Document(
                page_content=text,
                metadata={"source": str(path), "page": i, "type": "PDF_PAGE_TEXT"},
            )
        )
    return out


pages = read_pdf_pages(PDF_PATH)
len(pages), pages[0].metadata

(33,
 {'source': '../data/Understanding_Climate_Change.pdf',
  'page': 0,
  'type': 'PDF_PAGE_TEXT'})

In [6]:
splitter = RecursiveCharacterTextSplitter(chunk_size=900, chunk_overlap=120)
chunks = splitter.split_documents(pages)
for d in chunks:
    d.metadata["type"] = "PDF_TEXT_CHUNK"

len(chunks), chunks[0].metadata

(97,
 {'source': '../data/Understanding_Climate_Change.pdf',
  'page': 0,
  'type': 'PDF_TEXT_CHUNK'})

## 3) 建立（或加载）基础向量库

自适应检索的不同策略都会围绕同一个“基础语料向量库”工作（你也可以替换成你自己的知识库）。

In [7]:
VSTORE_DIR = Path("../vector_stores")
VSTORE_DIR.mkdir(parents=True, exist_ok=True)
BASE_STORE_DIR = VSTORE_DIR / "adaptive_retrieval_base_faiss"

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")


def load_or_build_base_store() -> FAISS:
    if BASE_STORE_DIR.exists():
        return FAISS.load_local(
            str(BASE_STORE_DIR),
            embeddings,
            allow_dangerous_deserialization=True,
        )
    store = FAISS.from_documents(chunks, embedding=embeddings)
    store.save_local(str(BASE_STORE_DIR))
    return store


store = load_or_build_base_store()
BASE_STORE_DIR

PosixPath('../vector_stores/adaptive_retrieval_base_faiss')

## 4) Query Classifier：把问题分成 4 类

我们用结构化输出（Pydantic）来让分类结果稳定可控。

In [8]:
QueryType = Literal["Factual", "Analytical", "Opinion", "Contextual"]


class QueryCategory(BaseModel):
    category: QueryType = Field(
        description="Query category: Factual, Analytical, Opinion, Contextual"
    )
    rationale: str = Field(
        description="简短理由（用于调试/理解分类），不要输出太长"
    )


classifier_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

classifier_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "你是一个问题分类器。将用户问题分类到 4 类：Factual/Analytical/Opinion/Contextual。"
            "只输出结构化结果。",
        ),
        (
            "human",
            "用户问题：{question}\n\n如果问题明显依赖用户背景/偏好/个人情况，选 Contextual；"
            "如果要求多角度或不同观点，选 Opinion；如果要求解释机制、综合分析，选 Analytical；"
            "否则选 Factual。",
        ),
    ]
)

classify_chain = classifier_prompt | classifier_llm.with_structured_output(
    QueryCategory,
    method="json_schema",
    strict=True,
)

classify_chain.invoke({"question": "温室气体为什么会导致变暖？"})

QueryCategory(category='Analytical', rationale='该问题要求解释温室气体导致变暖的机制，涉及综合分析其影响和作用原理。')

## 5) 通用工具：把 docs 拼成 context、以及一个“选择 top-k 索引”的轻量精排器

为了控制成本，我们尽量避免对每个 doc 单独调用一次 LLM，而是让 LLM 在候选列表里一次性选出 top-k 的索引。

In [9]:
def docs_to_context(docs: list[Document], max_chars: int = 1600) -> str:
    parts: list[str] = []
    for d in docs:
        page = d.metadata.get("page", "?")
        txt = d.page_content
        if len(txt) > max_chars:
            txt = txt[:max_chars] + "..."
        parts.append(f"[page={page}] {txt}")
    return "\n\n".join(parts)


class SelectedIndices(BaseModel):
    indices: list[int] = Field(description="选择的文档索引（升序或任意顺序均可）")


rank_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
rank_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "你是检索精排器。给你 query 和候选文档摘要列表。"
            "请选出最相关的 k 个候选文档索引。只输出结构化 indices。",
        ),
        (
            "human",
            "Query: {query}\n\nCandidates:\n{candidates}\n\nSelect top {k} indices.",
        ),
    ]
)

rank_chain = rank_prompt | rank_llm.with_structured_output(
    SelectedIndices, method="json_schema", strict=True
)


def select_topk(query: str, docs: list[Document], k: int) -> list[Document]:
    if not docs:
        return []
    k = max(1, min(k, len(docs)))
    candidates = "\n".join(
        [
            f"{i}: page={d.metadata.get('page')} | {d.page_content[:180].replace('\n',' ')}..."
            for i, d in enumerate(docs)
        ]
    )
    picked = rank_chain.invoke({"query": query, "candidates": candidates, "k": k}).indices
    picked = [i for i in picked if 0 <= i < len(docs)]
    if not picked:
        return docs[:k]
    # 去重保持顺序
    seen = set()
    out: list[Document] = []
    for i in picked:
        if i in seen:
            continue
        seen.add(i)
        out.append(docs[i])
        if len(out) >= k:
            break
    return out

## 6) 四种自适应检索策略

四种策略共享同一个 `store`，但会在“如何构造 query / 如何扩展 / 如何挑选证据”上做区别。

In [10]:
rewrite_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
rewrite_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "你是检索查询改写器。输出一句更适合向量检索的英文查询（简短、明确、包含关键实体）。",
        ),
        ("human", "原问题：{question}\n改写："),
    ]
)
rewrite_chain = rewrite_prompt | rewrite_llm | StrOutputParser()


subq_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)


class SubQueries(BaseModel):
    sub_queries: list[str] = Field(description="用于覆盖不同方面的子问题（英文更好检索）")


subq_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "你是分析型问题拆解器。将复杂问题拆成多个可检索的子问题（英文），覆盖不同方面。",
        ),
        ("human", "问题：{question}\n请生成 {n} 个子问题："),
    ]
)

subq_chain = subq_prompt | subq_llm.with_structured_output(
    SubQueries, method="json_schema", strict=True
)


vp_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)


class Viewpoints(BaseModel):
    viewpoints: list[str] = Field(description="不同视角/立场（英文短语）")


vp_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "你是观点挖掘器。给定一个主题/问题，列出若干不同视角（英文短语），用于按视角检索证据。",
        ),
        ("human", "问题：{question}\n请列出 {n} 个不同视角："),
    ]
)

vp_chain = vp_prompt | vp_llm.with_structured_output(
    Viewpoints, method="json_schema", strict=True
)


ctx_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
ctx_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "你是上下文化查询改写器。将用户上下文与问题融合，输出更适合检索的英文查询。",
        ),
        ("human", "User context: {user_context}\nQuestion: {question}\nContextualized query:"),
    ]
)
ctx_chain = ctx_prompt | ctx_llm | StrOutputParser()


def _dedup_docs(docs: list[Document]) -> list[Document]:
    seen = set()
    out: list[Document] = []
    for d in docs:
        key = (d.metadata.get("page"), d.page_content[:200])
        if key in seen:
            continue
        seen.add(key)
        out.append(d)
    return out


def retrieve_factual(question: str, k: int = 6, fetch_k: int = 18) -> list[Document]:
    enhanced = rewrite_chain.invoke({"question": question}).strip()
    candidates = store.similarity_search(enhanced, k=fetch_k)
    return select_topk(enhanced, candidates, k=k)


def retrieve_analytical(question: str, k: int = 6, n_subq: int = 4) -> list[Document]:
    subqs = subq_chain.invoke({"question": question, "n": n_subq}).sub_queries
    all_docs: list[Document] = []
    for sq in subqs:
        all_docs.extend(store.similarity_search(sq, k=3))
    all_docs = _dedup_docs(all_docs)
    # 让 LLM 从汇总候选里选出覆盖面更好的 top-k
    return select_topk(question, all_docs, k=k)


def retrieve_opinion(question: str, k: int = 6, n_vp: int = 4) -> list[Document]:
    vps = vp_chain.invoke({"question": question, "n": n_vp}).viewpoints
    all_docs: list[Document] = []
    for vp in vps:
        all_docs.extend(store.similarity_search(f"{question} | viewpoint: {vp}", k=3))
    all_docs = _dedup_docs(all_docs)
    return select_topk(question, all_docs, k=k)


def retrieve_contextual(question: str, user_context: str, k: int = 6, fetch_k: int = 18) -> list[Document]:
    cq = ctx_chain.invoke({"question": question, "user_context": user_context}).strip()
    candidates = store.similarity_search(cq, k=fetch_k)
    return select_topk(cq, candidates, k=k)

## 7) 总控：先分类，再选择策略检索

这里我们用一个 `RunnableLambda` 把“分类 + 分支策略”封装成一个可组合节点，便于后面接到 RAG 流水线里。

In [11]:
class AdaptiveInput(BaseModel):
    question: str
    user_context: str = ""


class AdaptiveRetrievalResult(BaseModel):
    category: QueryType
    rationale: str
    docs: list[Document]


def adaptive_retrieve(inp: dict) -> AdaptiveRetrievalResult:
    data = AdaptiveInput.model_validate(inp)
    cat = classify_chain.invoke({"question": data.question})
    if cat.category == "Factual":
        docs = retrieve_factual(data.question)
    elif cat.category == "Analytical":
        docs = retrieve_analytical(data.question)
    elif cat.category == "Opinion":
        docs = retrieve_opinion(data.question)
    else:
        docs = retrieve_contextual(data.question, user_context=data.user_context or "(none)")
    return AdaptiveRetrievalResult(category=cat.category, rationale=cat.rationale, docs=docs)


adaptive_retriever = RunnableLambda(adaptive_retrieve)

adaptive_retriever.invoke({"question": "解释温室效应的机制"}).category

'Analytical'

## 8) 最终回答：把检索结果变成 context，再生成答案

我们把“自适应检索”作为一个节点插入 RAG：输入是 `{question, user_context}`，输出是 `{category, rationale, docs}`，然后再生成答案。

In [12]:
qa_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

qa_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "你是一个严谨的问答助手。只根据给定的检索上下文回答；如果上下文不足以支撑结论，就明确说不知道。",
        ),
        (
            "human",
            "问题：{question}\n\n"
            "（系统判定问题类型：{category}，理由：{rationale}）\n\n"
            "检索上下文：\n{context}\n\n"
            "请给出中文回答：",
        ),
    ]
)


def result_to_prompt_vars(res: AdaptiveRetrievalResult) -> dict:
    return {
        "category": res.category,
        "rationale": res.rationale,
        "context": docs_to_context(res.docs),
    }


rag_chain = (
    {
        "question": RunnableLambda(lambda x: x["question"]).with_config(run_name="input_question"),
        "user_context": RunnableLambda(lambda x: x.get("user_context", "")),
    }
    | RunnableLambda(lambda x: {"question": x["question"], "user_context": x["user_context"]})
    | RunnableLambda(lambda x: {"input": x, "retrieval": adaptive_retriever.invoke(x)})
    | RunnableLambda(
        lambda x: {
            "question": x["input"]["question"],
            "category": x["retrieval"].category,
            "rationale": x["retrieval"].rationale,
            "context": docs_to_context(x["retrieval"].docs),
        }
    )
    | qa_prompt
    | qa_llm
    | StrOutputParser()
)

rag_chain.invoke({"question": "什么是温室效应？"})[:500]

'温室效应是指温室气体（如二氧化碳、甲烷和氧化亚氮）在大气中捕获来自太阳的热量，从而使地球保持足够温暖以支持生命的过程。虽然这一自然过程对生命至关重要，但人类活动的加剧（如燃烧化石燃料）使得这一过程变得更加强烈，导致气候变暖。'

## 9) 演示：四类问题各跑一遍

注意：我们的语料来自一份科普 PDF，所以“观点型问题”不一定能检索到真正对立观点；这里主要是演示自适应检索的编排方式。

In [13]:
examples = [
    {
        "name": "Factual",
        "question": "What are greenhouse gases?",
        "user_context": "",
    },
    {
        "name": "Analytical",
        "question": "Why does increasing CO2 lead to climate change? Explain the mechanism.",
        "user_context": "",
    },
    {
        "name": "Opinion",
        "question": "What are different perspectives on how to address climate change?",
        "user_context": "",
    },
    {
        "name": "Contextual",
        "question": "What should I do to reduce my carbon footprint?",
        "user_context": "I live in a big city, commute by car, and I can change my diet but not my job.",
    },
]

for ex in examples:
    print("\n===", ex["name"], "===")
    out = rag_chain.invoke({"question": ex["question"], "user_context": ex["user_context"]})
    print(out[:800])


=== Factual ===
温室气体是指能够吸收和发射红外辐射的气体，这些气体在大气中增加会导致温室效应，从而影响气候。主要的温室气体包括二氧化碳（CO2）、甲烷（CH4）和氧化亚氮（N2O）。这些气体能够捕获来自太阳的热量，保持地球的温暖，支持生命的存在。然而，人类活动的增加，特别是化石燃料的燃烧，导致了温室气体浓度的上升，从而加剧了气候变化。

=== Analytical ===
增加二氧化碳（CO2）导致气候变化的机制主要是通过增强温室效应来实现的。温室气体（如CO2、甲烷和氧化亚氮）能够吸收和重新辐射来自太阳的热量，从而使地球保持温暖。这一过程是自然的，维持了地球上生命的存在。

然而，人类活动，特别是燃烧化石燃料（如煤、石油和天然气）所释放的大量CO2，加剧了这一自然过程。自工业革命以来，化石燃料的消费显著增加，导致大气中CO2浓度上升。这种增加的CO2捕获了更多的热量，导致全球气温上升，从而引发气候变化。

此外，森林砍伐也加剧了这一问题，因为树木作为碳汇，能够吸收大气中的CO2。当树木被砍伐后，储存的碳会释放回大气，进一步加剧温室效应。因此，CO2的增加通过增强温室效应，导致气候变化。

=== Opinion ===
应对气候变化的不同观点包括：

1. **政策与法规**：各国通过实施碳定价、可再生能源激励和排放法规等政策来实现气候目标。这些国家战略需要与全球目标对齐，同时考虑地方需求和能力。

2. **地方行动**：地方政府和社区在气候行动中发挥着重要作用，包括城市规划、公共交通改善和社区保护等地方倡议。草根运动和公众意识提升活动也是推动地方变革的重要因素。

3. **研究与创新**：持续的研究和创新对于开发新技术和应对气候变化的策略至关重要。这包括可再生能源、碳捕集与储存以及可持续农业等领域的进展。

4. **整体性方法**：应对气候变化需要一种整体性的方法，整合环境、社会和经济维度。可持续发展、循环经济和生态公正是指导这一方法的关键原则。

5. **全球团结**：全球团结与合作是应对气候变化这一全球性挑战的基础。这包括支持脆弱国家和社区，分享资源和技术。

6. **文化运动**：文化运动在动员公众支持气候行动方面发挥着重要作用，如“未来星期五”和“灭绝叛乱”等运动通过创造性策略和直接行动推动变革。

7. **教育与青年参与